In [1]:
import sys
import pickle

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [ ]:
for try_idx in range(2):
    try:
        from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
        from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
        from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
        from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
        from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

        from src.qa_pipeline import QAPipelineConfig
        from src.qa_pipeline.query_parser import QueryLLMParserConfig
        from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

        from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
        from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
        from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

        from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

        from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

        from src.utils import Logger, ReaderMetrics
        from src.utils.data_structs import TripletCreator

        # other imports
    except RuntimeError as e:
        from pathlib import Path
        fix_path = Path("./import_fix.py")
        if fix_path.is_file():
            %run {fix_path} --base_dir BASEDIR
        else:
            raise e

#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [3]:
PKL_GRAPH_PATH = '../../data/pickled_graphs/DiaasqGigachat.pickle'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)

In [ ]:
length = 211542

print(len(formated_triplets))
formated_triplets = formated_triplets[:length]
print(len(formated_triplets))

#### 2. Задаём конфигурацию графа знаний

In [5]:
# in-memory storage
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
KV_STORAGE_CONFIG = KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)

In [ ]:
#
LANGUAGE = 'auto' # 'ru' , 'en', 'auto

#
RETRIEVER_NAME = 'astar' # 'astar', 'bfs'
RETRIEVER_HYPERP = AStarGraphSearchConfig(max_depth=10, max_passed_nodes=500) # AStarGraphSearchConfig, BFSSearchConfig 

# embedder hyperp
DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'

# vector dbs hyperp
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

In [ ]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    graph_struct_config=GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG),
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=NODES_DB_PATH, db_name='vectorized_nodes', need_to_clear=NEED_TO_CLEAR)),
        tripletsdb_driver_config=VectorDriverConfig(db_config=VectorDBConnectionConfig(
            path=TRIPLETS_DB_PATH, db_name='vectorized_triplets', need_to_clear=NEED_TO_CLEAR)),
        embedder_config=EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)),
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP,
            cache_config=KV_STORAGE_CONFIG),
        answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE)),
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(lang=LANGUAGE),
        updator_config=LLMUpdatorConfig(lang=LANGUAGE)),
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [ ]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

In [7]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 4. Добавляем в граф загруженные триплеты

In [ ]:
rkg_main.kg_model.graph_struct.create_triplets(formated_triplets)
formated_triplets_wid = [triplet for triplet in rkg_main.kg_model.graph_struct.db_conn.triplets_ids.values()]
rkg_main.kg_model.embeddings_struct.add_triplets(formated_triplets_wid)

#### 5. Q&A

In [9]:
qa_examples = [
  ("Matthew has positive, negative or neutral opinion about game of Mi 10pro on 22.12.2018?",
  "Positive"),
  ("Lily has positive, negative or neutral opinion about power of XiaoMi on 4.7.2019?",
  "Negative"),
  ("What opinion (positive, negative or neutral) about scheduling of IQOO9 was last during Zachary's experience of IQOO9?",
  "positive"),
  ("What opinion (positive, negative or neutral) about electricity of Apple was last during Jessica's experience of Apple?",
  "positive"),
  ("What Abraham's opinion (positive, negative or neutral) about fast charging of Xiaomi was dominant during using Xiaomi?",
  "positive"),
  ("Do Adrian and Herbert have any common devices (which Adrian and Herbert both use)? If so, list common devices. Otherwise, answer 'No'.",
  "No"),
  ("Do Bernard and Bailey have any common devices (which Bernard and Bailey both use)? If so, list common devices. Otherwise, answer 'No'.",
  "Apple"),
  ("Which people have negative opinion about battery of Apple phone on 15.9.2018?",
  "Hugh")
  ]

In [ ]:
for question in qa_examples:
    print("MODEL ANSWER: ", rkg_main.answer_question(question[0]))
    print("TRUE ANSWER: ", question[1])
    print("=" * 35)